In [2]:
import pandas as pd
import duckdb

In [3]:
!ls /kaggle/input

steam-dataset-2025-multi-modal-gaming-analytics


Cheking the input, whrther the data is imported or not

In [4]:
!ls /kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics

steam_dataset_2025_csv_package_v1
steam_dataset_2025_embeddings_package_v1
steam_dataset_2025_power_users_dump_v1
steam-dataset-2025-v1


Listing the directories

In [5]:
BASE_PATH = (
    "/kaggle/input/"
    "steam-dataset-2025-multi-modal-gaming-analytics/"
    "steam_dataset_2025_csv_package_v1/"
    "steam_dataset_2025_csv"
)

import os
os.listdir(BASE_PATH)

['application_platforms.csv',
 'application_publishers.csv',
 'application_genres.csv',
 'MANIFEST.json',
 'genres.csv',
 'categories.csv',
 'reviews.csv',
 'application_categories.csv',
 'developers.csv',
 'applications.csv',
 'publishers.csv',
 'platforms.csv',
 'application_developers.csv']

In [6]:
import pandas as pd

applications = pd.read_csv(f"{BASE_PATH}/applications.csv")
applications.shape

/tmp/ipykernel_55/1212156742.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  applications = pd.read_csv(f"{BASE_PATH}/applications.csv")


(239664, 30)

In [7]:
applications.head()

,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,...,mat_pc_os_min,mat_pc_processor_min,mat_pc_memory_min,mat_pc_graphics_min,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at
0,10,Counter-Strike,game,False,2000-11-01,0,Play the world's number 1 online action game. ...,"English<strong>*</strong>, French<strong>*</st...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
1,20,Team Fortress Classic,game,False,1999-04-01,0,One of the most popular online action games of...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
2,30,Day of Defeat,game,False,2003-05-01,0,Enlist in an intense brand of Axis vs. Allied ...,"English, French, German, Italian, Spanish - Spain",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
3,40,Deathmatch Classic,game,False,2001-06-01,0,Enjoy fast-paced multiplayer gaming with Death...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
4,50,Half-Life: Opposing Force,game,False,1999-11-01,0,Return to the Black Mesa Research Facility as ...,"English, French, German, Korean",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00


In [8]:
applications.columns

Index(['appid', 'name', 'type', 'is_free', 'release_date', 'required_age',
       'short_description', 'supported_languages', 'header_image',
       'background', 'metacritic_score', 'recommendations_total',
       'mat_supports_windows', 'mat_supports_mac', 'mat_supports_linux',
       'mat_initial_price', 'mat_final_price', 'mat_discount_percent',
       'mat_currency', 'mat_achievement_count', 'mat_pc_os_min',
       'mat_pc_processor_min', 'mat_pc_memory_min', 'mat_pc_graphics_min',
       'mat_pc_os_rec', 'mat_pc_processor_rec', 'mat_pc_memory_rec',
       'mat_pc_graphics_rec', 'created_at', 'updated_at'],
      dtype='object')

In [9]:
con = duckdb.connect()

In [10]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/application_genres.csv')
LIMIT 5
""").df()

,appid,genre_id
0,10,122
1,20,122
2,30,122
3,40,122
4,50,122


In [11]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/genres.csv')
LIMIT 5
""").df()

,id,name
0,1,Aventură
1,2,Многопользовательские игры
2,3,Nezávislé
3,4,Strategie
4,5,Strategy


In [12]:
con.execute(f"""
SELECT
    ag.appid,
    string_agg(g.name, ', ') AS genres
FROM read_csv_auto('{BASE_PATH}/application_genres.csv') ag
JOIN read_csv_auto('{BASE_PATH}/genres.csv') g
    ON ag.genre_id = g.id
GROUP BY ag.appid
LIMIT 5
""").df()

,appid,genres
0,220,Action
1,1230,Action
2,2700,"Strategy, Simulation"
3,3312,Casual
4,4102,Casual


the above code need to be removed

In [13]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_genres AS
SELECT
    ag.appid,
    string_agg(g.name, ', ') AS genres
FROM read_csv_auto('{BASE_PATH}/application_genres.csv') ag
JOIN read_csv_auto('{BASE_PATH}/genres.csv') g
    ON ag.genre_id = g.id
GROUP BY ag.appid
""")

In [15]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/application_categories.csv')
LIMIT 5
""").df()

,appid,category_id
0,10,72
1,10,90
2,10,130
3,10,133
4,10,193


In [16]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/categories.csv')
LIMIT 5
""").df()

,id,name
0,1,Remote Play en móvil
1,2,Flera spelare
2,3,Tablette Remote Play
3,4,Multiplayer
4,5,Таблицы лидеров Steam


In [17]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_categories AS
SELECT
    ac.appid,
    string_agg(c.name, ', ') AS categories
FROM read_csv_auto('{BASE_PATH}/application_categories.csv') ac
JOIN read_csv_auto('{BASE_PATH}/categories.csv') c
    ON ac.category_id = c.id
GROUP BY ac.appid
""")

In [18]:
con.execute("""
SELECT *
FROM app_categories
LIMIT 5
""").df()

,appid,categories
0,2458480,"Family Sharing, Single-player, Steam Cloud"
1,2458690,"Full controller support, Game demo"
2,2458820,Game demo
3,2458851,"Steam Achievements, In-App Purchases, MMO, Mul..."
4,2458860,"Steam Achievements, Steam Trading Cards, Famil..."


In [19]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_developers AS
SELECT
    ad.appid,
    string_agg(DISTINCT d.name, ', ') AS developers
FROM read_csv_auto('{BASE_PATH}/application_developers.csv') ad
JOIN read_csv_auto('{BASE_PATH}/developers.csv') d
    ON ad.developer_id = d.id
GROUP BY ad.appid
""")

In [20]:
con.execute("""
SELECT *
FROM app_developers
LIMIT 5
""").df()

,appid,developers
0,4900,Unknown Worlds Entertainment
1,31990,Her Interactive
2,34110,Sports Interactive
3,60340,MumboJumbo
4,206060,Spiderweb Software


In [21]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_publishers AS
SELECT
    ap.appid,
    string_agg(DISTINCT p.name, ', ') AS publishers
FROM read_csv_auto('{BASE_PATH}/application_publishers.csv') ap
JOIN read_csv_auto('{BASE_PATH}/publishers.csv') p
    ON ap.publisher_id = p.id
GROUP BY ap.appid
""")

In [22]:
con.execute("""
SELECT *
FROM app_publishers
LIMIT 5
""").df()

,appid,publishers
0,7670,2K
1,34200,SEGA
2,90207,Giants Software
3,263520,Plug In Digital
4,266090,Project Whitecard Studios Inc.


In [23]:
con.execute(f"""
CREATE OR REPLACE TABLE applications_wide AS
SELECT
    a.*,
    g.genres,
    c.categories,
    d.developers,
    p.publishers
FROM read_csv_auto(
        '{BASE_PATH}/applications.csv',
        ignore_errors=true
     ) a
LEFT JOIN app_genres g      ON a.appid = g.appid
LEFT JOIN app_categories c  ON a.appid = c.appid
LEFT JOIN app_developers d  ON a.appid = d.appid
LEFT JOIN app_publishers p  ON a.appid = p.appid
""")

In [24]:
con.execute("""
SELECT COUNT(*) FROM applications_wide
""").df()

,count_star()
0,239653


In [25]:
con.execute("""
SELECT * FROM applications_wide LIMIT 5
""").df()

,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,...,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at,genres,categories,developers,publishers
0,670490,Rise of Man,game,False,2017-09-15,0,Rise of Man is a pre historic strategy game wi...,"English<strong>*</strong>, Simplified Chinese<...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,Windows 7 / Windows 8,Intel Core i5 processor (or greater) 64bit,8 GB RAM,512 MB DirectX 10 compatible card,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:04:07.947948+00:00,"Strategy, Simulation, Indie, Early Access","Family Sharing, Single-player",Darkcross Games,Darkcross Games
1,670500,RC Plane 3,game,True,2017-08-07,0,Learn to fly a large selection of RC Planes in...,English<strong>*</strong><br><strong>*</strong...,https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,None,None,None,None,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:04:14.546455+00:00,"Simulation, Racing, Sports","Steam Achievements, PvP, Cross-Platform Multip...",FrozenPepper S.R.L,FrozenPepper S.R.L.
2,670510,ColorBlend FX: Desaturation,game,False,2024-04-18,0,Help Splatians blend the stolen colors back wi...,"English<strong>*</strong>, Bulgarian, Simplifi...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,Windows 10,i5 6th generation or new,8 GB RAM,nVidia GeForce GTX 1060 and up / AMD RX 580 an...,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:04:45.421414+00:00,"Indie, Adventure, Action","Steam Achievements, Full controller support, F...",Pi-Dev Bulgaria,Pi-Dev Bulgaria
3,670550,Light Biker Hockey,game,False,NaT,0,Do you like motorbikes and hockey too? Than th...,English<strong>*</strong><br><strong>*</strong...,https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,None,None,None,None,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:02:12.010655+00:00,"Racing, Sports, Adventure","PvP, In-App Purchases, Cross-Platform Multipla...",Theodor Niklas,Theodor Niklas
4,670560,3D MiniGolf: Candy Shop,dlc,True,2017-10-09,0,New DLC &quot;Candy Shopf&quot; available! Sug...,"English<strong>*</strong>, German<strong>*</st...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,10.8,2 GHz Intel-based and above,2 GB RAM,256 MB RAM required,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:02:12.010655+00:00,"Simulation, Casual, Sports","Steam Achievements, Family Sharing, Single-pla...",Z-Software GmbH,familyplay


In [26]:
df = con.execute("""
SELECT *
FROM applications_wide
""").df()

df.shape

(239653, 34)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239653 entries, 0 to 239652
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype                  
---  ------                 --------------   -----                  
 0   appid                  239653 non-null  int64                  
 1   name                   239653 non-null  object                 
 2   type                   236986 non-null  object                 
 3   is_free                239653 non-null  bool                   
 4   release_date           202799 non-null  datetime64[us]         
 5   required_age           239653 non-null  int64                  
 6   short_description      224162 non-null  object                 
 7   supported_languages    221995 non-null  object                 
 8   header_image           239653 non-null  object                 
 9   background             239653 non-null  object                 
 10  metacritic_score       5298 non-null    float64         

In [28]:
df.describe(include="all").T.head(34)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
appid,239653.0,NaN,NaN,NaN,2032205.26384,10.0,1122588.0,1993730.0,2946270.0,3996190.0,1068241.097208
name,239653,237753,Aurora,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,236986,5,game,150276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_free,239653,2,False,191381,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,202799,NaN,NaN,NaN,2021-10-08 08:04:34.510229,1969-12-31 00:00:00,2019-07-12 00:00:00,2022-06-03 00:00:00,2024-06-12 00:00:00,9998-12-31 00:00:00,NaN
required_age,239653.0,NaN,NaN,NaN,0.271376,0.0,0.0,0.0,0.0,120.0,2.115163
short_description,224162,206921,Embark on your adventure with player from all ...,284,NaN,NaN,NaN,NaN,NaN,NaN,NaN
supported_languages,221995,30502,English,54664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
header_image,239653,239565,https://shared.akamai.steamstatic.com/store_it...,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
background,239653,239653,https://store.akamai.steamstatic.com/images/st...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
df.to_csv("/kaggle/working/applications_wide.csv", index=False)

In [30]:
con.execute(f"""
SELECT DISTINCT recommendationid
FROM read_csv_auto('{BASE_PATH}/reviews.csv')
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,recommendationid
0,100002344
1,100008402
2,100011335
3,100018018
4,100021115
5,100027012
6,100030764
7,10003548
8,100045093
9,100049306


In [31]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW reviews_aggregated AS
SELECT
    appid,
    COUNT(*) AS n_reviews,

    -- sentiment
    SUM(CASE WHEN voted_up THEN 1 ELSE 0 END) AS positive_reviews,
    SUM(CASE WHEN NOT voted_up THEN 1 ELSE 0 END) AS negative_reviews,

    -- engagement (correct column)
    AVG(author_playtime_forever) AS avg_playtime,

    -- review usefulness
    AVG(votes_up) AS avg_helpful_votes

FROM read_csv_auto(
    '{BASE_PATH}/reviews.csv',
    ignore_errors=true
)
GROUP BY appid
""")


In [32]:
con.execute("""
SELECT *
FROM reviews_aggregated
LIMIT 10
""").df()

,appid,n_reviews,positive_reviews,negative_reviews,avg_playtime,avg_helpful_votes
0,49810,100,80.0,20.0,542.910000,3.060000
1,1741300,5,3.0,2.0,229.000000,3.600000
2,1625760,9,9.0,0.0,329.777778,1.444444
3,1595230,1,0.0,1.0,18.000000,7.000000
4,588880,18,16.0,2.0,182.555556,3.000000
5,881510,5,2.0,3.0,72.000000,8.800000
6,1707770,21,19.0,2.0,38165.142857,2.047619
7,278190,75,53.0,22.0,13293.280000,5.480000
8,1864320,5,0.0,5.0,26.600000,28.000000
9,1426770,15,13.0,2.0,0.000000,5.133333


In [33]:
con.execute("""
CREATE OR REPLACE TABLE applications_wide_with_reviews AS
SELECT
    a.*,

    -- review volume
    r.n_reviews,
    r.positive_reviews,
    r.negative_reviews,

    -- derived metric
    r.positive_reviews * 1.0 / NULLIF(r.n_reviews, 0) AS positive_ratio,

    -- engagement
    r.avg_playtime,
    r.avg_helpful_votes

FROM applications_wide a
LEFT JOIN reviews_aggregated r
    ON a.appid = r.appid
""")

In [34]:
con.execute("""
SELECT *
FROM applications_wide_with_reviews
LIMIT 3
""").df()

,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,...,genres,categories,developers,publishers,n_reviews,positive_reviews,negative_reviews,positive_ratio,avg_playtime,avg_helpful_votes
0,1367960,百詰怪軼與金魚,game,False,2022-01-13,0,唯美奇幻的多結局視覺小說。 ──現實與非現實的交界處，被遺忘的人們交織而成的物語。,Traditional Chinese,https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,"Casual, RPG","Family Sharing, Single-player",戀愛奶昔,威向文化,35,32.0,3.0,0.914286,691.485714,1.657143
1,1508280,2076 - Midway Multiverse,game,False,2022-02-03,0,Paying homage to classic side-scrolling shoote...,"English<strong>*</strong>, Spanish - Spain<str...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,"Simulation, Indie, Action","VR Only, Steam Achievements, Family Sharing, S...",Ivanovich Games,Ivanovich Games,38,31.0,7.0,0.815789,242.763158,2.763158
2,1279330,Under Lock,game,False,2021-01-12,0,Under Lock is a multiplayer asymmetrical PvP h...,"English, Russian, Simplified Chinese",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,"Indie, Early Access, Action","LAN PvP, Steam Achievements, PvP, VR Supported...",NioTum,NioTum,64,44.0,20.0,0.687500,382.968750,1.750000


In [35]:
df = con.execute("""
SELECT *
FROM applications_wide_with_reviews
""").df()

df.shape

(239653, 40)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239653 entries, 0 to 239652
Data columns (total 40 columns):
 #   Column                 Non-Null Count   Dtype                  
---  ------                 --------------   -----                  
 0   appid                  239653 non-null  int64                  
 1   name                   239653 non-null  object                 
 2   type                   236986 non-null  object                 
 3   is_free                239653 non-null  bool                   
 4   release_date           202799 non-null  datetime64[us]         
 5   required_age           239653 non-null  int64                  
 6   short_description      224162 non-null  object                 
 7   supported_languages    221995 non-null  object                 
 8   header_image           239653 non-null  object                 
 9   background             239653 non-null  object                 
 10  metacritic_score       5298 non-null    float64         

In [37]:
df.to_csv("/kaggle/working/applications_wide_with_reviews.csv", index=False)

In [38]:
missing_pct = df.isna().mean().sort_values(ascending=False)
missing_pct.head(40)

metacritic_score         0.977893
recommendations_total    0.905872
mat_achievement_count    0.743800
mat_pc_graphics_rec      0.615294
mat_pc_memory_rec        0.595987
mat_pc_processor_rec     0.594772
mat_pc_os_rec            0.574568
negative_reviews         0.510534
positive_reviews         0.510534
n_reviews                0.510534
positive_ratio           0.510534
avg_playtime             0.510534
avg_helpful_votes        0.510534
mat_discount_percent     0.395309
mat_currency             0.395309
mat_initial_price        0.395309
mat_final_price          0.395309
mat_pc_graphics_min      0.291910
mat_pc_memory_min        0.243606
mat_pc_processor_min     0.234639
mat_pc_os_min            0.182097
release_date             0.153781
genres                   0.132366
publishers               0.105536
supported_languages      0.073682
short_description        0.064639
categories               0.058422
developers               0.044581
type                     0.011129
appid         

In [39]:
df.isnull().sum().sort_values(ascending=False)

metacritic_score         234355
recommendations_total    217095
mat_achievement_count    178254
mat_pc_graphics_rec      147457
mat_pc_memory_rec        142830
mat_pc_processor_rec     142539
mat_pc_os_rec            137697
negative_reviews         122351
positive_reviews         122351
n_reviews                122351
positive_ratio           122351
avg_playtime             122351
avg_helpful_votes        122351
mat_discount_percent      94737
mat_currency              94737
mat_initial_price         94737
mat_final_price           94737
mat_pc_graphics_min       69957
mat_pc_memory_min         58381
mat_pc_processor_min      56232
mat_pc_os_min             43640
release_date              36854
genres                    31722
publishers                25292
supported_languages       17658
short_description         15491
categories                14001
developers                10684
type                       2667
appid                         0
name                          0
is_free 

In [40]:
eda_cols = [
    # identifiers
    'appid', 'name',

    # metadata
    'type', 'is_free', 'required_age', 'release_date',

    # pricing
    'mat_initial_price', 'mat_final_price', 'mat_discount_percent',

    # content
    'genres', 'categories', 'developers', 'publishers',

    # sentiment
    'n_reviews', 'positive_reviews', 'negative_reviews', 'positive_ratio',

    # engagement
    'avg_playtime', 'avg_helpful_votes',

    # platforms
    'mat_supports_windows', 'mat_supports_linux', 'mat_supports_mac'
]

df_eda = df[eda_cols]
df_eda.shape

(239653, 22)

In [41]:
df_eda.to_csv("/kaggle/working/Steam_data_after_EDA.csv", index=False)

In [42]:
df_eda.head(3)

,appid,name,type,is_free,required_age,release_date,mat_initial_price,mat_final_price,mat_discount_percent,genres,...,publishers,n_reviews,positive_reviews,negative_reviews,positive_ratio,avg_playtime,avg_helpful_votes,mat_supports_windows,mat_supports_linux,mat_supports_mac
0,1367960,百詰怪軼與金魚,game,False,0,2022-01-13,1399.0,1399.0,0.0,"Casual, RPG",...,威向文化,35,32.0,3.0,0.914286,691.485714,1.657143,True,True,True
1,1508280,2076 - Midway Multiverse,game,False,0,2022-02-03,1999.0,1999.0,0.0,"Simulation, Indie, Action",...,Ivanovich Games,38,31.0,7.0,0.815789,242.763158,2.763158,True,True,True
2,1279330,Under Lock,game,False,0,2021-01-12,999.0,999.0,0.0,"Indie, Early Access, Action",...,NioTum,64,44.0,20.0,0.687500,382.968750,1.750000,True,True,True


In [43]:
df_eda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239653 entries, 0 to 239652
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   appid                 239653 non-null  int64         
 1   name                  239653 non-null  object        
 2   type                  236986 non-null  object        
 3   is_free               239653 non-null  bool          
 4   required_age          239653 non-null  int64         
 5   release_date          202799 non-null  datetime64[us]
 6   mat_initial_price     144916 non-null  float64       
 7   mat_final_price       144916 non-null  float64       
 8   mat_discount_percent  144916 non-null  float64       
 9   genres                207931 non-null  object        
 10  categories            225652 non-null  object        
 11  developers            228969 non-null  object        
 12  publishers            214361 non-null  object        
 13 

In [44]:
eda_cols1 = [
    # identifiers
    'appid', 'name',

    # metadata
    'type', 'is_free', 'required_age', 'release_date',

    # pricing
    'mat_initial_price', 'mat_final_price', 'mat_discount_percent',

    # content
    'genres', 'categories', 'developers', 'publishers',

    # external / reputation signals (sparse but useful)
    'metacritic_score',
    'recommendations_total',

    # sentiment (core)
    'n_reviews',

    # engagement
    'avg_playtime',

    # platforms
    'mat_supports_windows', 'mat_supports_linux', 'mat_supports_mac'
]

df_eda1 = df[eda_cols1]
df_eda1.shape

(239653, 20)

In [45]:
df_eda1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239653 entries, 0 to 239652
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   appid                  239653 non-null  int64         
 1   name                   239653 non-null  object        
 2   type                   236986 non-null  object        
 3   is_free                239653 non-null  bool          
 4   required_age           239653 non-null  int64         
 5   release_date           202799 non-null  datetime64[us]
 6   mat_initial_price      144916 non-null  float64       
 7   mat_final_price        144916 non-null  float64       
 8   mat_discount_percent   144916 non-null  float64       
 9   genres                 207931 non-null  object        
 10  categories             225652 non-null  object        
 11  developers             228969 non-null  object        
 12  publishers             214361 non-null  obje

In [46]:
df_eda1.isnull().sum().sort_values(ascending=False)

metacritic_score         234355
recommendations_total    217095
n_reviews                122351
avg_playtime             122351
mat_final_price           94737
mat_initial_price         94737
mat_discount_percent      94737
release_date              36854
genres                    31722
publishers                25292
categories                14001
developers                10684
type                       2667
is_free                       0
appid                         0
name                          0
required_age                  0
mat_supports_windows          0
mat_supports_linux            0
mat_supports_mac              0
dtype: int64

In [47]:
df_eda1.to_csv("/kaggle/working/Steam_final_data.csv", index=False)